
# Amazon Search Ranking — Week 6 V4
## Final Evaluation, Ablation, and Cold-Start Generalization Report

### Goal
Consolidate Week 6 V1–V3 into one final evaluation.

This notebook does **not** train a new model. It focuses on rigorous final analysis:

1. Compare V1 / V2 / V3
2. Measure the contribution of the LLM relevance signal
3. Compare cold-start vs historical performance
4. Check whether the +15% target was achieved
5. Quantify the remaining gap to LambdaMART
6. Produce final plots and a ready-to-use Week 6 summary

### Required uploads
Upload these ZIP files together:

- `amazon_week6_v1_coldstart_benchmark_results.zip`
- `amazon_week6_v2_fixed_zero_shot_adapter_results.zip`
- `amazon_week6_v3_fast_llm_relevance_results.zip`

### Final output
`amazon_week6_v4_final_evaluation_results.zip`


In [ ]:

# =========================
# 0. SETUP
# =========================
import os, re, json, zipfile, shutil, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

ROOT = Path("/content/week6_v4")
UPLOAD_DIR = ROOT / "uploads"
EXTRACT_DIR = ROOT / "extracted"
OUT_DIR = ROOT / "outputs"

for p in [ROOT, UPLOAD_DIR, EXTRACT_DIR, OUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

TARGET_IMPROVEMENT_PCT = 15.0

print("Week 6 V4 environment ready.")


In [ ]:

# =========================
# 1. UPLOAD V1 + V2 FIXED + V3 FAST
# =========================
from google.colab import files

uploaded = files.upload()

for name, content in uploaded.items():
    (UPLOAD_DIR / name).write_bytes(content)

zip_files = list(UPLOAD_DIR.glob("*.zip"))

print("Uploaded:")
for p in zip_files:
    print(" -", p.name)

if len(zip_files) < 3:
    print("WARNING: expected 3 Week 6 ZIP files.")


In [ ]:

# =========================
# 2. EXTRACT + LOCATE FILES
# =========================
for zpath in zip_files:
    dest = EXTRACT_DIR / zpath.stem
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath, "r") as z:
        z.extractall(dest)

def find_file(name):
    hits = list(EXTRACT_DIR.rglob(name))
    return hits[0] if hits else None

paths = {
    "v1_cold_metrics": find_file("cold_start_benchmark_metrics.csv"),
    "v1_hist_metrics": find_file("historical_benchmark_metrics.csv"),
    "v1_definition": find_file("cold_start_definition.json"),
    "v2_holdout_metrics": find_file("zero_shot_holdout_metrics.csv"),
    "v2_full_metrics": find_file("zero_shot_full_coldstart_metrics.csv"),
    "v2_weights": find_file("zero_shot_best_weights.csv"),
    "v2_feature_diag": find_file("zero_shot_feature_diagnostics.csv"),
    "v3_holdout_metrics": find_file("v3_holdout_metrics.csv"),
    "v3_full_metrics": find_file("v3_full_coldstart_metrics.csv"),
    "v3_fusion": find_file("llm_fusion_search.csv"),
    "v3_llm_diag": find_file("llm_score_diagnostics.csv"),
    "v3_score_by_label": find_file("llm_score_by_label_diagnostic.csv"),
    "v3_pairs": find_file("v3_scored_pairs.csv"),
}

missing = [k for k, v in paths.items() if v is None]
print("Missing optional/required files:", missing)

if paths["v1_cold_metrics"] is None or paths["v2_holdout_metrics"] is None or paths["v3_holdout_metrics"] is None:
    raise RuntimeError("Required V1/V2/V3 metric files are missing.")


In [ ]:

# =========================
# 3. LOAD DATA
# =========================
v1_cold = pd.read_csv(paths["v1_cold_metrics"])
v1_hist = pd.read_csv(paths["v1_hist_metrics"]) if paths["v1_hist_metrics"] else pd.DataFrame()

v2_hold = pd.read_csv(paths["v2_holdout_metrics"])
v2_full = pd.read_csv(paths["v2_full_metrics"]) if paths["v2_full_metrics"] else pd.DataFrame()
v2_weights = pd.read_csv(paths["v2_weights"]) if paths["v2_weights"] else pd.DataFrame()
v2_diag = pd.read_csv(paths["v2_feature_diag"]) if paths["v2_feature_diag"] else pd.DataFrame()

v3_hold = pd.read_csv(paths["v3_holdout_metrics"])
v3_full = pd.read_csv(paths["v3_full_metrics"]) if paths["v3_full_metrics"] else pd.DataFrame()
v3_fusion = pd.read_csv(paths["v3_fusion"]) if paths["v3_fusion"] else pd.DataFrame()
v3_llm_diag = pd.read_csv(paths["v3_llm_diag"]) if paths["v3_llm_diag"] else pd.DataFrame()
v3_label_diag = pd.read_csv(paths["v3_score_by_label"]) if paths["v3_score_by_label"] else pd.DataFrame()
v3_pairs = pd.read_csv(paths["v3_pairs"]) if paths["v3_pairs"] else pd.DataFrame()

if paths["v1_definition"]:
    with open(paths["v1_definition"], "r") as f:
        v1_definition = json.load(f)
else:
    v1_definition = {}

print("Loaded all Week 6 result tables.")


In [ ]:

# =========================
# 4. BUILD FINAL HELD-OUT COMPARISON
# =========================
# V3 contains the most complete held-out model comparison.
final_holdout = v3_hold.copy()

# Ensure % vs BM25 exists.
if "ndcg10_improvement_vs_bm25_pct" not in final_holdout.columns:
    final_holdout["ndcg10_improvement_vs_bm25_pct"] = np.nan

    if "BM25" in set(final_holdout["model"]):
        bm = final_holdout[final_holdout["model"] == "BM25"].iloc[0]["ndcg@10"]
        final_holdout["ndcg10_improvement_vs_bm25_pct"] = (
            (final_holdout["ndcg@10"] - bm) / bm * 100
        )

final_holdout["target_15pct_achieved"] = (
    final_holdout["ndcg10_improvement_vs_bm25_pct"] >= TARGET_IMPROVEMENT_PCT
)

display(
    final_holdout.sort_values("ndcg@10", ascending=False)
)


In [ ]:

# =========================
# 5. V1 -> V2 -> V3 PROGRESSION
# =========================
def get_metric(df, model, metric):
    x = df[df["model"] == model]
    if x.empty or metric not in x.columns:
        return np.nan
    return float(x.iloc[0][metric])

bm25_v2 = get_metric(v2_hold, "BM25", "ndcg@10")
zs_v2 = get_metric(v2_hold, "ZeroShotAdapter", "ndcg@10")
llm_only_v3 = get_metric(v3_hold, "LLM-only", "ndcg@10")
zs_llm_v3 = get_metric(v3_hold, "ZeroShot+LLM", "ndcg@10")
lambda_v3 = get_metric(v3_hold, "LambdaMART", "ndcg@10")

progression = pd.DataFrame([
    {"stage": "BM25 baseline", "ndcg@10": bm25_v2},
    {"stage": "V2 ZeroShot", "ndcg@10": zs_v2},
    {"stage": "V3 LLM-only", "ndcg@10": llm_only_v3},
    {"stage": "V3 ZeroShot+LLM", "ndcg@10": zs_llm_v3},
    {"stage": "LambdaMART upper reference", "ndcg@10": lambda_v3},
])

progression["absolute_gain_vs_bm25"] = progression["ndcg@10"] - bm25_v2
progression["relative_gain_vs_bm25_pct"] = (
    (progression["ndcg@10"] / bm25_v2) - 1
) * 100

display(progression)


In [ ]:

# =========================
# 6. LLM ABLATION
# =========================
# Core ablation:
# ZeroShotAdapter -> LLM-only -> ZeroShot+LLM

ablation_models = ["BM25", "ZeroShotAdapter", "LLM-only", "ZeroShot+LLM", "LambdaMART"]

ablation = final_holdout[
    final_holdout["model"].isin(ablation_models)
][[
    "model", "ndcg@10", "ndcg@3", "recall@3", "mrr",
    "ndcg10_improvement_vs_bm25_pct"
]].copy()

ablation = ablation.sort_values("ndcg@10", ascending=False)

display(ablation)

zs_row = ablation[ablation["model"] == "ZeroShotAdapter"]
llm_row = ablation[ablation["model"] == "LLM-only"]
fusion_row = ablation[ablation["model"] == "ZeroShot+LLM"]

if not zs_row.empty and not fusion_row.empty:
    v2_to_v3_abs = float(
        fusion_row.iloc[0]["ndcg@10"] - zs_row.iloc[0]["ndcg@10"]
    )
    v2_to_v3_rel = float(
        (fusion_row.iloc[0]["ndcg@10"] / zs_row.iloc[0]["ndcg@10"] - 1) * 100
    )

    print(f"V2 -> V3 absolute NDCG@10 gain: {v2_to_v3_abs:+.4f}")
    print(f"V2 -> V3 relative NDCG@10 gain: {v2_to_v3_rel:+.2f}%")


In [ ]:

# =========================
# 7. COLD-START VS HISTORICAL TRADEOFF
# =========================
# V1 historical/cold-start comparison provides the cleanest historical tradeoff view.
tradeoff_models = sorted(
    set(v1_cold["model"]).intersection(set(v1_hist["model"]))
) if not v1_hist.empty else []

tradeoff_rows = []

for model in tradeoff_models:
    c = v1_cold[v1_cold["model"] == model].iloc[0]
    h = v1_hist[v1_hist["model"] == model].iloc[0]

    tradeoff_rows.append({
        "model": model,
        "historical_ndcg@10": h["ndcg@10"],
        "cold_start_ndcg@10": c["ndcg@10"],
        "ndcg_gap_cold_minus_hist": c["ndcg@10"] - h["ndcg@10"],
        "historical_mrr": h["mrr"],
        "cold_start_mrr": c["mrr"],
        "mrr_gap_cold_minus_hist": c["mrr"] - h["mrr"],
    })

tradeoff = pd.DataFrame(tradeoff_rows)

if not tradeoff.empty:
    display(tradeoff.sort_values("cold_start_ndcg@10", ascending=False))
else:
    print("Historical comparison not available.")


In [ ]:

# =========================
# 8. FINAL TARGET ANALYSIS
# =========================
bm25 = get_metric(final_holdout, "BM25", "ndcg@10")
fusion = get_metric(final_holdout, "ZeroShot+LLM", "ndcg@10")
lamb = get_metric(final_holdout, "LambdaMART", "ndcg@10")

required_for_15 = bm25 * 1.15 if pd.notna(bm25) else np.nan

target_analysis = pd.DataFrame([{
    "bm25_ndcg@10": bm25,
    "target_ndcg@10_for_15pct": required_for_15,
    "zero_shot_plus_llm_ndcg@10": fusion,
    "zero_shot_plus_llm_vs_bm25_pct": ((fusion / bm25) - 1) * 100,
    "gap_to_15pct_target": fusion - required_for_15,
    "lambdamart_ndcg@10": lamb,
    "lambdamart_vs_bm25_pct": ((lamb / bm25) - 1) * 100,
    "fusion_gap_to_lambdamart": fusion - lamb,
}])

display(target_analysis)


In [ ]:

# =========================
# 9. LLM SIGNAL QUALITY SUMMARY
# =========================
if not v3_label_diag.empty:
    print("LLM score by relevance label:")
    display(v3_label_diag)

if not v3_llm_diag.empty:
    print("\nLLM score discrimination diagnostics:")
    display(v3_llm_diag)

if not v3_fusion.empty:
    best_fusion = v3_fusion.sort_values(
        ["objective", "ndcg@10"],
        ascending=False
    ).iloc[0]

    print("\nBest tuning fusion:")
    print("ZeroShot weight:", best_fusion["alpha_zero_shot"])
    print("LLM weight:", best_fusion["alpha_llm"])


In [ ]:

# =========================
# 10. V2 FEATURE CONTRIBUTION SUMMARY
# =========================
if not v2_weights.empty:
    print("V2 Zero-Shot adapter learned weights:")
    display(v2_weights.sort_values("weight", ascending=False))

if not v2_diag.empty:
    print("\nV2 feature diagnostics:")
    display(v2_diag)


In [ ]:

# =========================
# 11. FINAL PLOTS
# =========================
# Plot 1: Held-out model comparison
p = final_holdout.dropna(subset=["ndcg@10"]).sort_values("ndcg@10", ascending=False)

plt.figure(figsize=(10, 5))
plt.bar(p["model"], p["ndcg@10"])
plt.axhline(bm25 * 1.15, linestyle="--", linewidth=1.5, label="+15% target")
plt.ylabel("NDCG@10")
plt.xlabel("Model")
plt.title("Week 6 Final Held-Out Cold-Start Ranking Performance")
plt.xticks(rotation=35, ha="right")
plt.legend()
plt.tight_layout()
plt.savefig(
    OUT_DIR / "week6_v4_final_ndcg10_comparison.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()

# Plot 2: V1 -> V2 -> V3 progression
plt.figure(figsize=(9, 5))
plt.plot(
    progression["stage"],
    progression["ndcg@10"],
    marker="o"
)
plt.ylabel("NDCG@10")
plt.xlabel("Stage")
plt.title("Week 6 Cold-Start Ranking Progression")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(
    OUT_DIR / "week6_v4_progression.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()

# Plot 3: Ablation
plt.figure(figsize=(9, 5))
plt.bar(ablation["model"], ablation["ndcg@10"])
plt.ylabel("NDCG@10")
plt.xlabel("Ablation / model")
plt.title("Week 6 LLM Ablation")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(
    OUT_DIR / "week6_v4_llm_ablation.png",
    dpi=180,
    bbox_inches="tight"
)
plt.show()

# Plot 4: Historical vs cold-start
if not tradeoff.empty:
    plot_trade = tradeoff.set_index("model")[[
        "historical_ndcg@10",
        "cold_start_ndcg@10"
    ]]

    plot_trade.plot(kind="bar", figsize=(9, 5))
    plt.ylabel("NDCG@10")
    plt.xlabel("Model")
    plt.title("Historical vs Cold-Start NDCG@10")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(
        OUT_DIR / "week6_v4_historical_vs_coldstart.png",
        dpi=180,
        bbox_inches="tight"
    )
    plt.show()


In [ ]:

# =========================
# 12. AUTOMATIC FINAL CONCLUSIONS
# =========================
bm25_row = final_holdout[final_holdout["model"] == "BM25"].iloc[0]
fusion_row = final_holdout[final_holdout["model"] == "ZeroShot+LLM"].iloc[0]
lambda_row = final_holdout[final_holdout["model"] == "LambdaMART"].iloc[0]
zs_row = final_holdout[final_holdout["model"] == "ZeroShotAdapter"].iloc[0]
llm_row = final_holdout[final_holdout["model"] == "LLM-only"].iloc[0]

lines = []

lines.append("Amazon Search Ranking — Week 6 Final Evaluation")
lines.append("=" * 68)
lines.append("")
lines.append("Cold-start definition")
lines.append(
    f"- Type: {v1_definition.get('type', 'unknown')}"
)
lines.append(
    f"- Cold-start products: {v1_definition.get('n_products_cold_start', 'n/a')} "
    f"of {v1_definition.get('n_products_total', 'n/a')}"
)
lines.append(
    f"- Cold-start product share: "
    f"{v1_definition.get('cold_start_product_share', np.nan):.2%}"
    if isinstance(v1_definition.get('cold_start_product_share', None), (int, float))
    else "- Cold-start product share: n/a"
)

lines.append("")
lines.append("Held-out ranking results")
lines.append(
    f"- BM25 NDCG@10: {bm25_row['ndcg@10']:.4f}"
)
lines.append(
    f"- V2 ZeroShot NDCG@10: {zs_row['ndcg@10']:.4f}"
)
lines.append(
    f"- LLM-only NDCG@10: {llm_row['ndcg@10']:.4f}"
)
lines.append(
    f"- V3 ZeroShot+LLM NDCG@10: {fusion_row['ndcg@10']:.4f}"
)
lines.append(
    f"- LambdaMART NDCG@10: {lambda_row['ndcg@10']:.4f}"
)

lines.append("")
lines.append("Key improvements")
lines.append(
    f"- V2 ZeroShot vs BM25: "
    f"{zs_row['ndcg10_improvement_vs_bm25_pct']:+.2f}%"
)
lines.append(
    f"- V3 ZeroShot+LLM vs BM25: "
    f"{fusion_row['ndcg10_improvement_vs_bm25_pct']:+.2f}%"
)
lines.append(
    f"- V3 absolute gain over V2: "
    f"{fusion_row['ndcg@10'] - zs_row['ndcg@10']:+.4f} NDCG@10"
)
lines.append(
    f"- Remaining gap to LambdaMART: "
    f"{fusion_row['ndcg@10'] - lambda_row['ndcg@10']:+.4f} NDCG@10"
)

lines.append("")
lines.append("Target assessment")
lines.append(
    f"- +15% target NDCG@10 threshold: {bm25_row['ndcg@10']*1.15:.4f}"
)
lines.append(
    f"- Best history-independent result: {fusion_row['ndcg@10']:.4f}"
)
lines.append(
    "- +15% target achieved: "
    + ("YES" if fusion_row["ndcg10_improvement_vs_bm25_pct"] >= 15 else "NO")
)
lines.append(
    "- Important context: LambdaMART itself reaches "
    f"{lambda_row['ndcg10_improvement_vs_bm25_pct']:+.2f}% vs BM25."
)

lines.append("")
lines.append("Main conclusion")
lines.append(
    "- LLM relevance substantially closes the cold-start gap. "
    "The final ZeroShot+LLM model uses no historical behavioral signals "
    "yet performs very close to LambdaMART."
)
lines.append(
    "- The Week 6 +15% target was not reached, but the final model achieved "
    f"{fusion_row['ndcg10_improvement_vs_bm25_pct']:+.2f}% over BM25 and "
    f"finished only {abs(fusion_row['ndcg@10'] - lambda_row['ndcg@10']):.4f} "
    "NDCG@10 behind LambdaMART."
)
lines.append(
    "- Contrastive fine-tuning remains a reasonable stretch goal rather than "
    "a required Week 6 deliverable."
)

summary_text = "\n".join(lines)
print(summary_text)

(OUT_DIR / "week6_v4_final_summary.txt").write_text(summary_text)


In [ ]:

# =========================
# 13. SAVE TABLES
# =========================
final_holdout.to_csv(
    OUT_DIR / "week6_v4_final_holdout_metrics.csv",
    index=False
)

progression.to_csv(
    OUT_DIR / "week6_v4_model_progression.csv",
    index=False
)

ablation.to_csv(
    OUT_DIR / "week6_v4_llm_ablation.csv",
    index=False
)

target_analysis.to_csv(
    OUT_DIR / "week6_v4_target_analysis.csv",
    index=False
)

if not tradeoff.empty:
    tradeoff.to_csv(
        OUT_DIR / "week6_v4_historical_vs_coldstart.csv",
        index=False
    )

if not v2_weights.empty:
    v2_weights.to_csv(
        OUT_DIR / "week6_v4_zero_shot_weights.csv",
        index=False
    )

if not v3_label_diag.empty:
    v3_label_diag.to_csv(
        OUT_DIR / "week6_v4_llm_score_by_label.csv",
        index=False
    )

print("Saved final Week 6 tables.")


In [ ]:

# =========================
# 14. PACKAGE + DOWNLOAD
# =========================
zip_path = ROOT / "amazon_week6_v4_final_evaluation_results.zip"

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT_DIR.rglob("*"):
        if p.is_file():
            z.write(p, arcname=p.relative_to(OUT_DIR))

print("Created:", zip_path)
print("Size:", f"{zip_path.stat().st_size / 1024 / 1024:.2f} MB")

from google.colab import files
files.download(str(zip_path))



## Week 6 completion criteria

After this notebook runs, Week 6 has all required core deliverables:

- Cold-start subset definition
- Cold-start benchmark
- Zero-shot ranking adapter
- LLM relevance feature experiment
- Final cold-start evaluation
- Ablation analysis
- Historical-vs-cold-start comparison

Contrastive fine-tuning can be documented as **stretch / future work** unless you explicitly want to continue with an additional V5 experiment.
